# Site-D Ablation — no-D vs with-D

Compares the three **no-D arm** runs (Ceftriaxone, Ciprofloxacin, Ceftazidime)
against their archived **with-D runs**, under matched configuration (splits,
seed, hyperparameters, thresholds, rounds).

- Primary endpoint: per-site :term:`Balanced Accuracy` of **FedAvg (MLP)** on the
  shared A/B/C test sets (identical splits) — Δ = no-D − with-D.
- Site **D**: evaluated but never trained in the no-D arm → unseen-site transfer,
  compared with the with-D FedAvg and the archived cross-site A→D baselines.
- Secondary: Centralized MLP/RF (pooled A/B/C vs pooled A/B/C/D), FedAvg LR, FedRF.

Outputs land in `Site-D-Ablation/comparison_results/`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd()
if not (HERE / "Ceftriaxone").exists():
    HERE = Path("/home/asd/Projects/Flower/DataSet/Dryad-DataSet/Analysis/08-Federated-mlp-lr-rf/Site-D-Ablation")
ROOT = HERE.parent
OUT = HERE / "comparison_results"; OUT.mkdir(exist_ok=True)

ARCHIVE = {"Ceftriaxone": "07-Run", "Ciprofloxacin": "04-Run", "Ceftazidime": "03-Run"}
METHODS = ["Centralized MLP", "Centralized RF", "FL FedAvg (MLP)", "FL FedAvg (LR)", "FL FedRF (Trees)"]
SITES = ["A", "B", "C", "D"]

rows = []
for drug, run in ARCHIVE.items():
    wd = pd.read_csv(ROOT / "Drugs" / drug / run / "results" / "final_results.csv")
    nd = pd.read_csv(HERE / drug / "results" / "final_results.csv")
    for m in METHODS:
        wr = wd[wd["Method"] == m]
        nr = nd[nd["Method"] == m]
        if len(wr) == 0 or len(nr) == 0:
            print(f"  skip {drug}/{m} (missing in one arm)")
            continue
        wr, nr = wr.iloc[0], nr.iloc[0]
        for s in SITES:
            rows.append({"drug": drug, "method": m, "site": s,
                         "with_D": wr[f"{s}_BalAcc"], "no_D": nr[f"{s}_BalAcc"],
                         "delta": nr[f"{s}_BalAcc"] - wr[f"{s}_BalAcc"]})
cmp = pd.DataFrame(rows)
cmp.to_csv(OUT / "comparison_per_site.csv", index=False)
print(f"comparison rows: {len(cmp)}")
cmp.head()


In [ ]:
# ── Primary: FedAvg (MLP) per-site ──
fav = cmp[cmp["method"] == "FL FedAvg (MLP)"]
tab = fav.pivot(index="drug", columns="site", values=["with_D", "no_D", "delta"]).round(4)
tab = tab.reindex(columns=pd.MultiIndex.from_product([["with_D", "no_D", "delta"], SITES]))
print("FedAvg (MLP) — per-site Balanced Accuracy (with-D vs no-D)")
display(tab)
tab.to_csv(OUT / "comparison_fedavg_deltas.csv")

print("\nMean Δ across drugs (no-D − with-D), all methods:")
mean_delta = cmp.groupby(["method", "site"])["delta"].mean().unstack().round(4)
display(mean_delta)

for m in METHODS:
    sub = cmp[cmp["method"] == m]
    if len(sub) == 0:
        continue
    print(f"\n{m} — Δ (no-D − with-D):")
    display(sub.pivot(index="drug", columns="site", values="delta").round(4))


In [ ]:
# ── Site D when it is only evaluated (unseen-site transfer) ──
transfer_rows = []
for drug, run in ARCHIVE.items():
    wd = pd.read_csv(ROOT / "Drugs" / drug / run / "results" / "final_results.csv")
    nd = pd.read_csv(HERE / drug / "results" / "final_results.csv")
    fav_w = wd[wd["Method"] == "FL FedAvg (MLP)"].iloc[0]
    fav_n = nd[nd["Method"] == "FL FedAvg (MLP)"].iloc[0]
    row = {"drug": drug,
           "FedAvg_MLP_withD": fav_w["D_BalAcc"],
           "FedAvg_MLP_noD": fav_n["D_BalAcc"],
           "FedAvg_MLP_noD_AUC": fav_n["D_AUC"]}
    for m in ["Cross-Site MLP", "Cross-Site RF"]:
        r = wd[wd["Method"] == m]
        if len(r):
            row[m] = r.iloc[0]["D_BalAcc"]
    transfer_rows.append(row)
transfer = pd.DataFrame(transfer_rows).round(4)
print("Site D — Balanced Accuracy: trained (with-D) vs unseen (no-D) vs cross-site transfer")
display(transfer)
transfer.to_csv(OUT / "comparison_siteD_transfer.csv", index=False)

print("\nno-D arm: A/B/C-pooled (AllABC) vs 4-site All (D untrained) — FedAvg (MLP)")
allabc = []
for drug in ARCHIVE:
    nd = pd.read_csv(HERE / drug / "results" / "final_results.csv")
    r = nd[nd["Method"] == "FL FedAvg (MLP)"].iloc[0]
    allabc.append({"drug": drug, "AllABC_BalAcc": r.get("AllABC_BalAcc", np.nan),
                   "All_BalAcc": r["All_BalAcc"], "D_BalAcc": r["D_BalAcc"]})
display(pd.DataFrame(allabc).round(4))


In [ ]:
# ── Figure 1: per-site FedAvg (MLP), with-D vs no-D ──
drugs = list(ARCHIVE.keys())
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)
x = np.arange(len(SITES)); w = 0.38
for ax, drug in zip(axes, drugs):
    sub = cmp[(cmp["drug"] == drug) & (cmp["method"] == "FL FedAvg (MLP)")].set_index("site").reindex(SITES)
    ax.bar(x - w/2, sub["with_D"], w, label="with D (archived run)", color="#1f77b4", alpha=0.85)
    ax.bar(x + w/2, sub["no_D"], w, label="no D (ablation arm)", color="#d62728", alpha=0.85)
    for xi, dv in enumerate(sub["delta"]):
        ax.annotate(f"{dv:+.3f}", (xi + w/2, sub["no_D"].iloc[xi]), ha="center", va="bottom", fontsize=7.5)
    ax.set_xticks(x); ax.set_xticklabels([f"Site {s}" for s in SITES])
    ax.set_title(f"{drug} (mirror: {ARCHIVE[drug]})", fontsize=10)
    ax.grid(True, axis="y", ls="--", lw=0.4, alpha=0.5); ax.set_ylim(0.4, 1.0)
axes[0].set_ylabel("Balanced Accuracy")
axes[0].legend(fontsize=8, loc="lower right")
fig.suptitle("FedAvg (MLP) per-site Balanced Accuracy — site D dropped from training (Δ = no-D − with-D)")
fig.tight_layout()
fig.savefig(OUT / "noD_fedavg_per_site.pdf", bbox_inches="tight")
fig.savefig(OUT / "noD_fedavg_per_site.svg", bbox_inches="tight")
plt.show()

# ── Figure 2: site D transfer ──
fig, ax = plt.subplots(figsize=(9.5, 4.6))
x = np.arange(len(drugs)); w = 0.2
series = [("FedAvg_MLP_withD", "FedAvg MLP — trained on D (with-D)", "#1f77b4"),
          ("FedAvg_MLP_noD", "FedAvg MLP — D unseen (no-D)", "#d62728"),
          ("Cross-Site MLP", "Cross-site A→D MLP (archived)", "#7f7f7f"),
          ("Cross-Site RF", "Cross-site A→D RF (archived)", "#c7c7c7")]
for k, (col, label, color) in enumerate(series):
    if col not in transfer.columns:
        continue
    ax.bar(x + (k - 1.5) * w, transfer[col], w, label=label, color=color)
ax.set_xticks(x); ax.set_xticklabels(drugs, fontsize=9)
ax.set_ylabel("Site D Balanced Accuracy"); ax.set_ylim(0.4, 0.9)
ax.axhline(0.5, color="gray", ls="--", lw=0.8)
ax.set_title("Site D: trained vs unseen (no-D) vs cross-site transfer")
ax.grid(True, axis="y", ls="--", lw=0.4, alpha=0.5); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "noD_siteD_transfer.pdf", bbox_inches="tight")
fig.savefig(OUT / "noD_siteD_transfer.svg", bbox_inches="tight")
plt.show()
